# A DMD reversion signal, and the diagnostic that killed it

**Every number here comes from synthetic data generated inside the repository. It has no economic meaning.**

This is the cleaned-up version of a two-week research project. The idea was simple: forecast the next return with Dynamic Mode Decomposition, buy whatever is priced below its forecast, sell whatever is above. Four variants were tested (close and open prices, long and short).

It did not work. The interesting part is why, and the diagnostic in section 8 is the part worth copying: a forecast can look statistically significant and still have no directional content at all.

## 1. The idea

A model produces a forecast of the next return for every stock. If the price sits below the forecast, the stock is cheap relative to the model and should be bought; if it sits above, it should be sold.

The forecast comes from **Dynamic Mode Decomposition**: take the last 252 sessions of returns for a cross-section of stocks as a matrix, fit the linear operator that maps one day's cross-section to the next, and apply it to the last day.

The question is not whether the idea sounds sensible. It does. The question is whether the forecast predicts anything, and whether enough of the predicted move survives trading costs.

## 2. Setup

The code lives in the `dmd_reversion` package next to this notebook: `synthetic.py` makes the data, `dmd.py` is the forecast, `pipeline.py` has the thresholds and the portfolio, `diagnostics.py` is the test that matters. Run the notebook from the repository root.

In [1]:
import sys
from pathlib import Path

# run this notebook from the repository root
sys.path.insert(0, str(Path.cwd()))

import numpy as np
from dmd_reversion.synthetic import make_market
from dmd_reversion.dmd import dmd_forecast
from dmd_reversion.pipeline import WINDOW, HOLD, COST, WIN, run_variants, format_table

print("numpy", np.__version__, "| window", WINDOW, "| hold", HOLD,
      "| cost", COST, "| win", WIN)


numpy 2.3.5 | window 252 | hold 20 | cost 0.02 | win 0.05


## 3. Synthetic data

The generator makes a market with a common factor, sector factors and idiosyncratic noise. Switching on `signal="reversal"` makes today's idiosyncratic move partly reverse tomorrow, which gives the model a real structure to find.

In [2]:
def lag1_autocorr(x):
    return float(np.corrcoef(x[:-1].ravel(), x[1:].ravel())[0, 1])

clean, _, _ = make_market(n_tickers=150, n_days=900, seed=7, signal="none")
planted, _, _ = make_market(n_tickers=150, n_days=900, seed=7, signal="reversal")

print("shape (days, tickers):", clean.shape)
print("noise     lag-1 autocorrelation %+0.3f" % lag1_autocorr(clean))
print("reversal  lag-1 autocorrelation %+0.3f" % lag1_autocorr(planted))


shape (days, tickers): (900, 150)
noise     lag-1 autocorrelation +0.008
reversal  lag-1 autocorrelation -0.250


## 4. The first thing to check: does the operator learn, or memorise?

Full-rank DMD fits the window exactly whenever there are more stocks than transitions. With 300 names and 99 transitions the operator has 99 degrees of freedom for 99 transitions, so the in-sample fit is perfect and the forecast is meaningless.

With fewer names than transitions the same fit is over-determined and therefore regularised. That asymmetry is the most important thing to understand about this model class.

In [3]:
def in_sample_r2(window, rank):
    centered = window - window.mean(axis=0)
    x_d, y_d = centered[:-1].T, centered[1:].T
    u, s, vt = np.linalg.svd(x_d, full_matrices=False)
    r = len(s) if rank is None else int(min(rank, len(s)))
    ur = u[:, :r]
    op = ur.T @ y_d @ vt[:r].T @ np.diag(1.0 / np.maximum(s[:r], 1e-12))
    predicted = ur @ (op @ (ur.T @ x_d))
    denom = np.sum((y_d - y_d.mean(axis=0)) ** 2)
    return float(1.0 - np.sum((y_d - predicted) ** 2) / denom)

rng = np.random.default_rng(1)
for label, window in (("300 names, 99 transitions", rng.normal(size=(100, 300))),
                      ("25 names, 59 transitions", rng.normal(size=(60, 25)))):
    print(label)
    for rank in (None, 20, 5):
        print("   rank=%-5s in-sample R2 = %.4f"
              % ("full" if rank is None else rank, in_sample_r2(window, rank)))


300 names, 99 transitions
   rank=full  in-sample R2 = 1.0000
   rank=20    in-sample R2 = 0.0738
   rank=5     in-sample R2 = 0.0046
25 names, 59 transitions
   rank=full  in-sample R2 = 0.4449
   rank=20    in-sample R2 = 0.3395
   rank=5     in-sample R2 = -0.0080


## 5. The pipeline, in one call

Everything after the forecast is one function: three thresholds found on the validation period, then a portfolio on the test period with one position per stock, a 20-session hold, a 2% cost, and a 5% win threshold.

In [4]:
from dmd_reversion.pipeline import run_variants, format_table

# nothing else to set up: run_variants(signal, quick, rank, seed) does the four
# variants in one call -- forecast, threshold search on validation, portfolio on test
print("pipeline ready")


pipeline ready


## 6. Four variants on noise

With nothing planted, the thresholds still select something every day, so the pipeline always produces trades. The question is whether they are better than nothing.

Watch what happens: one variant will often land above zero. With a few dozen trades that is what luck looks like, and it is the reason a positive backtest on its own proves nothing.

In [5]:
rows_noise = run_variants("none", quick=True)
print(format_table(rows_noise))


variant       trades      win     gross       net
-------------------------------------------------
close_long        38    28.9%     0.51%    -1.49%
close_short       51    43.1%     3.34%     1.34%
open_long        114    29.8%    -0.69%    -2.69%
open_short       136    35.3%     0.16%    -1.84%


## 7. Four variants with a planted reversal

Now the data contains a real one-day reversal. The model can find it:

The table still looks unimpressive, and it should. At this sample size the backtest is dominated by noise in both directions, which is precisely why the next section exists.

In [6]:
rows_planted = run_variants("reversal", quick=True)
print(format_table(rows_planted))


variant       trades      win     gross       net
-------------------------------------------------
close_long       144    29.2%     0.86%    -1.14%
close_short      169    31.4%     1.27%    -0.73%
open_long         50    32.0%     0.30%    -1.70%
open_short        59    27.1%     0.22%    -1.78%


## 8. The diagnostic that matters

Ignore the size of the forecast and ask only whether its sign matches the realised sign, across every stock and every test day. Near 50% means the model has no directional content, whatever the backtest shows.

In [7]:
def panel_diagnostics(signal):
    close_ret, _, _ = make_market(n_tickers=150, n_days=900, seed=7, signal=signal)
    preds, actuals = [], []
    for t in range(WINDOW, 880, 5):
        preds.append(dmd_forecast(close_ret[t - WINDOW:t], rank=20))
        actuals.append(close_ret[t:t + HOLD].sum(axis=0))
    preds, actuals = np.concatenate(preds), np.concatenate(actuals)
    ok = np.isfinite(preds) & np.isfinite(actuals)
    accuracy = float(np.mean(np.sign(preds[ok]) == np.sign(actuals[ok])))
    a = np.argsort(np.argsort(preds[ok])).astype(float)
    b = np.argsort(np.argsort(actuals[ok])).astype(float)
    return accuracy, float(np.corrcoef(a, b)[0, 1])

print("%-10s%20s%12s" % ("data", "direction accuracy", "rank IC"))
print("-" * 42)
for name in ("none", "reversal"):
    accuracy, ic = panel_diagnostics(name)
    print("%-10s%19.2f%%%12.4f" % (name, 100 * accuracy, ic))


data        direction accuracy     rank IC
------------------------------------------
none                    50.59%      0.0003
reversal                51.43%      0.0282


## 9. The arithmetic that decides everything

A win nets +3% after the cost; a loss costs -7%. Break-even is therefore a 70% win rate, and no amount of tuning closes a gap of that size:

In [8]:
break_even = (COST + WIN) / (2 * WIN)
print("win threshold %.0f%% gross, cost %.0f%%" % (100 * WIN, 100 * COST))
print("a win nets %+.0f%%, a loss costs %+.0f%%" % (100 * (WIN - COST), 100 * (-WIN - COST)))
print("break-even win rate: %.0f%%" % (100 * break_even))
print()
for name, rows in (("noise", rows_noise), ("reversal", rows_planted)):
    print("%-9s best gross %+0.2f%%   best net %+0.2f%%"
          % (name, 100 * max(r["gross"] for r in rows), 100 * max(r["net"] for r in rows)))


win threshold 5% gross, cost 2%
a win nets +3%, a loss costs -7%
break-even win rate: 70%

noise     best gross +3.34%   best net +1.34%
reversal  best gross +1.27%   best net -0.73%


## 10. What to take away

- **Check direction before size.** One line of code, before any portfolio work.
- **The cost contract decides the problem.** With a 2% cost and a 5% target you need a 70% win rate. Argue about the contract before tuning the model.
- **Full-rank DMD is an interpolator whenever names exceed days.**
- **A magnitude can be informative while the sign is not.** In the original study the absolute forecast ranked future volatility usefully (rank IC 0.29-0.33) while its sign was useless.

What this notebook does **not** show: anything about real markets. The generator is a toy and no configuration here should be traded.

## 11. Repository layout

| path | what |
|---|---|
| `notebook.ipynb` | this walkthrough |
| `dmd_reversion/synthetic.py` | the data generator |
| `dmd_reversion/dmd.py` | the forecast (18 lines) |
| `dmd_reversion/pipeline.py` | thresholds, portfolio, the four variants |
| `dmd_reversion/diagnostics.py` | sign accuracy and rank IC |
| `run_demo.py` | the same four variants from the command line |
| `tests/test_pipeline.py` | 14 checks |